In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
from pytorch_implementation.prediction.vip3d.config import debug_forward_config
from pytorch_implementation.prediction.vip3d.model import VIP3DLite, compute_ade_fde

cfg = debug_forward_config()
model = VIP3DLite(cfg).eval()

def _build_debug_batch(cfg):
    torch.manual_seed(13)
    batch_size = 2
    num_agents = 5

    base_xy = torch.randn(batch_size, num_agents, 2)
    velocity = torch.randn(batch_size, num_agents, 2) * 0.2
    history_time = torch.arange(cfg.history_steps, dtype=torch.float32).view(1, 1, cfg.history_steps, 1)
    future_time = torch.arange(1, cfg.future_steps + 1, dtype=torch.float32).view(1, 1, cfg.future_steps, 1)

    history_xy = base_xy.unsqueeze(2) + history_time * velocity.unsqueeze(2)
    history_vel = velocity.unsqueeze(2).expand(-1, -1, cfg.history_steps, -1)
    agent_history = torch.cat([history_xy, history_vel], dim=-1)

    agent_valid = torch.ones(batch_size, num_agents, cfg.history_steps, dtype=torch.bool)
    agent_valid[0, 0, -1] = False
    agent_valid[1, 3, 0] = False

    map_polylines = torch.randn(
        batch_size,
        cfg.map_tokens,
        cfg.map_points_per_token,
        cfg.map_input_dim,
    )

    last_observed = history_xy[:, :, -1]
    gt_future = last_observed.unsqueeze(2) + future_time * velocity.unsqueeze(2)
    gt_valid = torch.ones(batch_size, num_agents, cfg.future_steps, dtype=torch.bool)
    gt_valid[0, 2, -2:] = False
    return agent_history, map_polylines, agent_valid, gt_future, gt_valid

agent_history, map_polylines, agent_valid, gt_future, gt_valid = _build_debug_batch(cfg)

print(f"Model: {type(model).__name__}")
print(f"Config: {cfg.name}")
print(f"agent_history: {tuple(agent_history.shape)}")
print(f"map_polylines: {tuple(map_polylines.shape)}")
print(f"agent_valid: {tuple(agent_valid.shape)}")

In [ ]:
from typing import Any

def _first_tensor(value: Any):
    """Extract the first tensor from a nested structure."""
    import torch
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            t = _first_tensor(item)
            if t is not None:
                return t
    if isinstance(value, dict):
        for item in value.values():
            t = _first_tensor(item)
            if t is not None:
                return t
    return None

def _iter_tensors(value: Any):
    """Iterate over all tensors in a nested structure."""
    import torch
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _iter_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _iter_tensors(item)

def _register_hook(module, name: str, capture: dict, handles: list) -> None:
    """Register a forward hook that stores output in capture[name]."""
    def _hook(_module, _inputs, output):
        capture[name] = output
    handles.append(module.register_forward_hook(_hook))

def _print_shape(label: str, value) -> None:
    """Print shape of first tensor in value."""
    t = _first_tensor(value)
    if t is not None:
        print(f"  {label}: {tuple(t.shape)}")
    else:
        print(f"  {label}: <no tensor>")

def _check_finite(capture: dict, outputs: dict) -> None:
    """Assert all captured intermediates and outputs are finite."""
    import torch
    for name, value in capture.items():
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in {name}"
    for name, value in outputs.items():
        if value is None:
            continue
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in output {name}"
    print("All intermediate and final tensors are finite.")

# VIP3D Paper-to-Code (Prediction)

## 0) Scope and Artifacts

- Model key: `prediction/vip3d`
- Task: trajectory prediction
- Implementation: `pytorch_implementation/prediction/vip3d/`
- Test: `tests/prediction/vip3d.py`
- Notebook: `study/notebook/prediction/vip3d_paper_to_code.ipynb`
- Paper/source: external reference (local PDF not currently present): [VIP3D paper](https://arxiv.org/abs/2203.08376)

This reimplementation is a strict-parity, pure-PyTorch forward path that keeps the core trajectory-prediction structure:
1) temporal agent encoding,
2) map token encoding,
3) agent-map fusion,
4) multimodal future decoding,
5) ADE/FDE metric smoke validation.

## Chunk 0: End-to-End Prediction Contract

### Goal
Predict `M` candidate trajectories for each agent over a future horizon `T_f`, plus mode confidence scores.

### Interface
- Input history tensor: `H \in R^{B x A x T_h x F_a}`
- Input map tensor: `P \in R^{B x N_m x S_m x F_m}`
- Output trajectories: `Y_hat \in R^{B x A x M x T_f x 2}`
- Output mode logits: `L \in R^{B x A x M}`
- Output mode probs: `pi = softmax(L)`

### Code Mapping
- `H` -> `agent_history` in `VIP3DLite.forward`
- `P` -> `map_polylines` in `VIP3DLite.forward`
- `Y_hat` -> `outputs["trajectories"]`
- `pi` -> `outputs["mode_probs"]`

### Sanity Check
- `mode_probs.sum(-1) == 1` for each `(b, a)`.
- `trajectories.shape[-2] == future_steps` (horizon integrity).

In [ ]:
# Chunk 0: End-to-end prediction contract
with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)

print("=== Output shapes ===")
for key, val in outputs.items():
    if torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")

## Chunk 1: Agent Temporal Encoding

### Goal
Encode each agent history token sequence into a compact latent token.

### Equations
`z_{b,a,t} = W_h h_{b,a,t} + b_h`  
`u_{b,a,1:T_h} = TransformerEncoder(z_{b,a,1:T_h}, mask)`  
`q_{b,a} = Mean_t(u_{b,a,t}, valid_mask)`

### Symbol Table
- `h_{b,a,t}`: raw motion features (x, y, vx, vy)
- `z_{b,a,t}`: projected history token
- `u_{b,a,t}`: temporally encoded token
- `q_{b,a}`: agent latent token

### Code Mapping
- `W_h` -> `history_input_proj`
- `TransformerEncoder` -> `history_encoder`
- masked mean -> `_masked_temporal_mean`

### Sanity Check
- `history_tokens` preserves time axis `[B, A, T_h, C]`.
- Masked pooling never divides by zero (`clamp(min=1)`).

In [ ]:
# Chunk 1: Agent temporal encoding
capture, handles = {}, []
_register_hook(model.history_input_proj, "history.input_proj", capture, handles)
_register_hook(model.history_encoder.layers[0], "history.encoder.layer0", capture, handles)
_register_hook(model.history_norm, "history.norm", capture, handles)

with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
for h in handles:
    h.remove()

print("=== Agent temporal encoding ===")
_print_shape("history.input_proj", capture["history.input_proj"])
_print_shape("history.encoder.layer0", capture["history.encoder.layer0"])
_print_shape("history.norm", capture["history.norm"])
_print_shape("history_tokens", outputs["history_tokens"])

## Chunk 2: Map Tokenization

### Goal
Convert polyline points into fixed map tokens for cross-attention.

### Equations
`m_{b,n,s} = MLP_map(p_{b,n,s})`  
`k_{b,n} = W_k ( (1/S_m) * sum_s m_{b,n,s} )`

### Symbol Table
- `p_{b,n,s}`: map point feature
- `m_{b,n,s}`: encoded map point
- `k_{b,n}`: map token

### Code Mapping
- `MLP_map` -> `map_point_mlp`
- token projection `W_k` -> `map_token_proj`

### Sanity Check
- `map_tokens.shape == [B, N_m, C]`.
- All map tensors pass finite checks in `tests/prediction/vip3d.py`.

In [ ]:
# Chunk 2: Map tokenization
capture, handles = {}, []
_register_hook(model.map_point_mlp, "map.point_mlp", capture, handles)
_register_hook(model.map_token_proj, "map.token_proj", capture, handles)

with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
for h in handles:
    h.remove()

print("=== Map tokenization ===")
_print_shape("map.point_mlp", capture["map.point_mlp"])
_print_shape("map.token_proj", capture["map.token_proj"])
_print_shape("map_tokens", outputs["map_tokens"])

## Chunk 3: Agent-Map Fusion

### Goal
Fuse each agent token with map context via cross-attention.

### Equation
`f_{b,a} = LN( q_{b,a} + CrossAttn(query=q_{b,a}, key=k_{b,*}, value=k_{b,*}) )`

### Symbol Table
- `q_{b,a}`: agent token
- `k_{b,*}`: all map tokens in scene
- `f_{b,a}`: fused token for trajectory decoding

### Code Mapping
- `CrossAttn` -> `agent_map_attention`
- residual + normalization -> `fusion_norm`

### Sanity Check
- `fused_tokens.shape == [B, A, C]`.
- Attention output is captured with hooks (`fusion.cross_attention`).

In [ ]:
# Chunk 3: Agent-map fusion
capture, handles = {}, []
_register_hook(model.agent_map_attention, "fusion.cross_attention", capture, handles)
_register_hook(model.fusion_norm, "fusion.norm", capture, handles)

with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
for h in handles:
    h.remove()

print("=== Agent-map fusion ===")
_print_shape("fusion.cross_attention", capture["fusion.cross_attention"])
_print_shape("fusion.norm", capture["fusion.norm"])
_print_shape("fused_tokens", outputs["fused_tokens"])
_print_shape("attention_weights", outputs["attention_weights"])

## Chunk 4: Multimodal Trajectory Decoding

### Goal
Decode fused tokens into mode scores and future 2D trajectories.

### Equations
`r_{b,a} = DecoderMLP(f_{b,a})`  
`L_{b,a,:} = W_mode r_{b,a}`  
`Delta_{b,a,m,1:T_f} = reshape(W_delta r_{b,a})`  
`Y_hat_{b,a,m,t} = x_{b,a}^{last} + sum_{tau=1..t} Delta_{b,a,m,tau}`

### Symbol Table
- `r_{b,a}`: decoder latent
- `L_{b,a,:}`: mode logits
- `Delta`: per-step trajectory deltas
- `x^{last}`: last valid observed position
- `Y_hat`: absolute predicted trajectory

### Code Mapping
- decoder trunk -> `decoder.trunk`
- mode logits -> `decoder.mode_head`
- deltas -> `decoder.delta_head`
- cumulative integration -> `deltas.cumsum(dim=3)`
- best mode index -> `outputs["best_mode"]`

### Sanity Check
- Trajectory consistency:
  `Y_hat[..., t] - Y_hat[..., t-1] == Delta[..., t]` for `t >= 1`.

In [ ]:
# Chunk 4: Multimodal trajectory decoding
capture, handles = {}, []
_register_hook(model.decoder.trunk, "decoder.trunk", capture, handles)
_register_hook(model.decoder.mode_head, "decoder.mode_head", capture, handles)
_register_hook(model.decoder.delta_head, "decoder.delta_head", capture, handles)

with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
for h in handles:
    h.remove()

print("=== Decoder outputs ===")
_print_shape("decoder.trunk", capture["decoder.trunk"])
_print_shape("decoder.mode_head", capture["decoder.mode_head"])
_print_shape("decoder.delta_head", capture["decoder.delta_head"])
_print_shape("mode_probs", outputs["mode_probs"])
_print_shape("trajectories", outputs["trajectories"])
_print_shape("best_trajectory", outputs["best_trajectory"])

assert torch.allclose(outputs["mode_probs"].sum(dim=-1), torch.ones_like(outputs["mode_probs"].sum(dim=-1)), atol=1e-5)
batch_size, num_agents = outputs["best_mode"].shape
gather_idx = outputs["best_mode"].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1).expand(
    batch_size, num_agents, 1, cfg.future_steps, 2
)
expected_best = outputs["trajectories"].gather(dim=2, index=gather_idx).squeeze(2)
assert torch.allclose(outputs["best_trajectory"], expected_best, atol=1e-5)
print("best-mode trajectory contract check passed.")

## Chunk 5: Prediction Metrics (ADE/FDE Smoke)

### Goal
Validate that trajectory metrics are computable and finite.

### Equations
`d_{b,a,m,t} = ||Y_hat_{b,a,m,t} - Y_{b,a,t}||_2`  
`ADE_{b,a,m} = (1/T_valid) * sum_t d_{b,a,m,t}`  
`FDE_{b,a,m} = d_{b,a,m,t_last_valid}`  
`mode* = argmin_m FDE_{b,a,m}`

### Code Mapping
- metric function -> `compute_ade_fde(...)`
- best-of-K selection -> `argmin` over per-mode FDE

### Sanity Check
- `ade` and `fde` are scalar, finite, non-negative.
- Horizon and validity masks are respected (`gt_valid`).

In [ ]:
# Chunk 5: Prediction metrics (ADE/FDE smoke)
with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
metrics = compute_ade_fde(
    trajectories=outputs["trajectories"],
    gt_future=gt_future,
    gt_valid=gt_valid,
)
print("metric smoke:", {"ade": float(metrics["ade"]), "fde": float(metrics["fde"])})

## Implementation Notes

- This module intentionally avoids MMDet3D/MMCV runtime dependencies.
- The forward path is deterministic in debug config (`dropout=0.0`).
- The test file validates:
  - intermediate hook captures,
  - shape contracts at major boundaries,
  - finite-value checks,
  - prediction-specific horizon and metric checks.

In [ ]:
# Final finite check with major hooks
capture, handles = {}, []
_register_hook(model.history_input_proj, "history.input_proj", capture, handles)
_register_hook(model.history_encoder.layers[0], "history.encoder.layer0", capture, handles)
_register_hook(model.history_norm, "history.norm", capture, handles)
_register_hook(model.map_point_mlp, "map.point_mlp", capture, handles)
_register_hook(model.map_token_proj, "map.token_proj", capture, handles)
_register_hook(model.agent_map_attention, "fusion.cross_attention", capture, handles)
_register_hook(model.fusion_norm, "fusion.norm", capture, handles)
_register_hook(model.decoder.trunk, "decoder.trunk", capture, handles)
_register_hook(model.decoder.mode_head, "decoder.mode_head", capture, handles)
_register_hook(model.decoder.delta_head, "decoder.delta_head", capture, handles)

with torch.no_grad():
    outputs = model(agent_history, map_polylines, agent_valid)
for h in handles:
    h.remove()

_check_finite(capture, outputs)